In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!git clone https://github.com/FarazAbdulMuqtader/Urdu_NLP_Chatbot.git
%cd Urdu_NLP_Chatbot

Cloning into 'Urdu_NLP_Chatbot'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 73 (delta 39), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 3.80 MiB | 12.20 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/kaggle/working/Urdu_NLP_Chatbot


In [3]:
%cd /kaggle/working/Urdu_NLP_Chatbot
!git pull

/kaggle/working/Urdu_NLP_Chatbot
Already up to date.


In [4]:
!ls

01_explore_data.py  04_evaluate.py		     README.md
02_preprocess.py    05_demo.py			     roman_urdu_clean.csv
03_train_model.py   notebook-urdu-nlp-chatbot.ipynb


In [5]:
!cat 01_explore_data.py

from datasets import load_dataset
import pandas as pd

# Load dataset
dataset = load_dataset("Khubaib01/RomanUrdu-NLP-Sentiment-Corpus")

# Convert to pandas
df = pd.DataFrame(dataset['train'])

print("=== BASIC INFO ===")
print(f"Total samples: {len(df)}")
print(f"\nColumn names: {df.columns.tolist()}")

print("\n=== LABEL DISTRIBUTION ===")
print(df['label'].value_counts())

print("\n=== SAMPLE TEXTS PER LABEL ===")
for label in df['label'].unique():
    sample = df[df['label'] == label]['message'].iloc[0]
    print(f"\n{label}: {sample}")

print("\n=== TEXT LENGTH STATS ===")
print(f"Average word count: {df['word_length'].mean():.1f}")
print(f"Shortest message: {df['word_length'].min():.0f} words")
print(f"Longest message: {df['word_length'].max():.0f} words")

print("\n=== FIRST 5 ROWS ===")
print(df[['message', 'label']].head())

In [6]:
!python 01_explore_data.py

README.md: 5.75kB [00:00, 1.65MB/s]
RomanUrdu_NLP_Sentiment-Corpus.csv: 100%|██| 11.7M/11.7M [00:01<00:00, 7.04MB/s]
Generating train split: 100%|█| 134053/134053 [00:00<00:00, 370758.78 examples/s
=== BASIC INFO ===
Total samples: 134053

Column names: ['message', 'label', 'char_length', 'word_length']

=== LABEL DISTRIBUTION ===
label
Negative    53445
Neutral     43212
Positive    37396
Name: count, dtype: int64

=== SAMPLE TEXTS PER LABEL ===

Positive: imran khan ke lye bahut acha hoga

Neutral: Market maker😂

Negative: pmln tarha videos bna ecp or media pr dan

=== TEXT LENGTH STATS ===
Average word count: 13.6
Shortest message: 0 words
Longest message: 3212 words

=== FIRST 5 ROWS ===
                                             message     label
0                  imran khan ke lye bahut acha hoga  Positive
1                                      Market maker😂   Neutral
2  bohut achay kamal ki sharing hai peer o murshi...  Positive
3          pmln tarha videos bna ecp or media p

In [7]:
!cat 02_preprocess.py

import pandas as pd
import re
from datasets import load_dataset

# ── 1. Load ───────────────────────────────────────────────
dataset = load_dataset("Khubaib01/RomanUrdu-NLP-Sentiment-Corpus")
df = pd.DataFrame(dataset['train'])
original_len = len(df)
print("Label distribution before cleaning: ")
print(df["label"].value_counts())

print(f"Before cleaning: {len(df)} rows")

# ── 2. Clean function ─────────────────────────────────────
def clean_text(text):
    # Handle NaN/empty values
    if not isinstance(text, str):
        return ''
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtag symbol but keep word
    text = re.sub(r'#', '', text)
    # Remove emojis, punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Lowercase
    text = text.lower()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── 3. Apply cleaning ───────────────────────────────

In [8]:
!python 02_preprocess.py

Label distribution before cleaning: 
label
Negative    53445
Neutral     43212
Positive    37396
Name: count, dtype: int64
Before cleaning: 134053 rows
Duplicates: 30990
After cleaning: 129377 rows
Removed: 4676 rows
Label distribution after cleaning: 
label
Negative    52518
Neutral     40601
Positive    36258
Name: count, dtype: int64

=== BEFORE vs AFTER ===

Original : imran khan ke lye bahut acha hoga
Cleaned  : imran khan ke lye bahut acha hoga
Label    : Positive

Original : Market maker😂
Cleaned  : market maker
Label    : Neutral

Original : bohut achay kamal ki sharing hai peer o murshid ki asliat ahista ahista khul kr samne aa rahi hai very good
Cleaned  : bohut achay kamal ki sharing hai peer o murshid ki asliat ahista ahista khul kr samne aa rahi hai very good
Label    : Positive

✅ Saved to roman_urdu_clean.csv
Duplicates: 28783
After cleaning: 129377 rows
Removed: 4676 rows
Label distribution after cleaning: 
label
Negative    52518
Neutral     40601
Positive    36258
Nam

In [9]:
!cat 03_train_model.py

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from torch.optim import AdamW

# Reproducibility
torch.manual_seed(42)

# ── 1. Load CSV ───────────────────────────────────────────
df = pd.read_csv('roman_urdu_clean.csv')

# ── 2. Encode labels ──────────────────────────────────────
label_map = {'Positive': 0, 'Negative': 1, 'Neutral': 2}
df['label_id'] = df['label'].map(label_map)
df = df.dropna(subset=['label_id'])
df['label_id'] = df['label_id'].astype(int)

# FIX: removed the df.sample(3000, ...) test-run limit — we now train on
# the full cleaned dataset (~129K rows), which is what the README claims.
print(f"Using {len(df)} rows for training")

# ── 3. Split 80/20 ────────────────────────────────────────
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['label_id']
)
print(f"Train: 

In [10]:
!python 03_train_model.py

Using 129377 rows for training
Train: 103501 | Test: 25876
config.json: 100%|█████████████████████████████| 615/615 [00:00<00:00, 2.74MB/s]
tokenizer_config.json: 100%|██████████████████| 25.0/25.0 [00:00<00:00, 145kB/s]
sentencepiece.bpe.model: 100%|█████████████| 5.07M/5.07M [00:00<00:00, 28.4MB/s]
tokenizer.json: 9.10MB [00:00, 19.0MB/s]
Train batches: 6469
Test batches : 1618

Loading model...
model.safetensors: 100%|████████████████████| 1.12G/1.12G [00:04<00:00, 264MB/s]
Loading weights: 100%|█| 197/197 [00:00<00:00, 1326.46it/s, Materializing param=
XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_

In [11]:
!python 04_evaluate.py

Evaluating on 25876 held-out rows
Loading weights: 100%|█| 201/201 [00:00<00:00, 1370.83it/s, Materializing param=
Evaluating on: cuda

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

    Positive       0.81      0.82      0.82      7252
    Negative       0.84      0.85      0.85     10504
     Neutral       0.78      0.76      0.77      8120

    accuracy                           0.81     25876
   macro avg       0.81      0.81      0.81     25876
weighted avg       0.81      0.81      0.81     25876


=== CONFUSION MATRIX ===
Rows = actual label, Columns = predicted label
          Positive  Negative  Neutral
Positive      5977       508      767
Negative       588      8920      996
Neutral        802      1177     6141

=== MISCLASSIFIED EXAMPLES (4838 total) ===
                                                                                                                                                                     text true_label pr

In [12]:
!python 05_demo.py

Loading weights: 100%|█| 201/201 [00:00<00:00, 1435.69it/s, Materializing param=
Model loaded. Running on: cuda

=== Roman Urdu Sentiment Demo ===

Text       : ye movie bohut acha tha
Prediction : Positive (99.7% confidence)
All scores : Positive: 99.7%, Negative: 0.1%, Neutral: 0.3%

Text       : mujhe bilkul pasand nahi aya
Prediction : Negative (99.6% confidence)
All scores : Positive: 0.1%, Negative: 99.6%, Neutral: 0.3%

Text       : kal market gaya tha
Prediction : Neutral (79.5% confidence)
All scores : Positive: 2.9%, Negative: 17.6%, Neutral: 79.5%

Text       : bohut bura din tha aj
Prediction : Negative (99.6% confidence)
All scores : Positive: 0.1%, Negative: 99.6%, Neutral: 0.3%

Text       : chalo choro is baat ko
Prediction : Neutral (71.2% confidence)
All scores : Positive: 24.8%, Negative: 4.0%, Neutral: 71.2%

Text       : wah kya baat hai zabardast
Prediction : Positive (99.5% confidence)
All scores : Positive: 99.5%, Negative: 0.1%, Neutral: 0.4%



In [22]:
from huggingface_hub import login
login()

In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('./trained_model')
tokenizer = AutoTokenizer.from_pretrained('./trained_model')

model.push_to_hub("FarazAbdulMuqtader/roman-urdu-sentiment-xlmr")
tokenizer.push_to_hub("FarazAbdulMuqtader/roman-urdu-sentiment-xlmr")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/FarazAbdulMuqtader/roman-urdu-sentiment-xlmr/commit/0716e9394afb6389f3eb29821781654af768a278', commit_message='Upload tokenizer', commit_description='', oid='0716e9394afb6389f3eb29821781654af768a278', pr_url=None, repo_url=RepoUrl('https://huggingface.co/FarazAbdulMuqtader/roman-urdu-sentiment-xlmr', endpoint='https://huggingface.co', repo_type='model', repo_id='FarazAbdulMuqtader/roman-urdu-sentiment-xlmr'), pr_revision=None, pr_num=None)